# Multi-task Gaussian processes

This notebook compares `MultiTaskGP` and `KroneckerMultiTaskGP`.

- `MultiTaskGP` uses **long-format** data: the task index is included as a feature.
- `KroneckerMultiTaskGP` uses a **block design**: every task is observed at the same `X`, and `Y` has one column per task.

Use `MultiTaskGP` when tasks may have different observation locations. Use `KroneckerMultiTaskGP` when all tasks are measured at the same design points.


## 1. Imports and reproducibility


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import KroneckerMultiTaskGP, MultiTaskGP

torch.manual_seed(0)
dtype = torch.double


## 2. Synthetic correlated tasks


In [ ]:
def task0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)

def task1(x: torch.Tensor) -> torch.Tensor:
    return 0.7 * torch.sin(2 * torch.pi * x + 0.35) + 0.25

x0 = torch.linspace(0.05, 0.95, 10, dtype=dtype).unsqueeze(-1)
x1 = torch.linspace(0.10, 0.90, 7, dtype=dtype).unsqueeze(-1)

y0 = task0(x0) + 0.03 * torch.randn_like(x0)
y1 = task1(x1) + 0.03 * torch.randn_like(x1)


## 3. `MultiTaskGP`: long-format representation

The task feature is appended as the final column. Here task 0 and task 1 deliberately use different `x` locations.


In [ ]:
task0_col = torch.zeros(len(x0), 1, dtype=dtype)
task1_col = torch.ones(len(x1), 1, dtype=dtype)

train_X_long = torch.cat(
    [
        torch.cat([x0, task0_col], dim=-1),
        torch.cat([x1, task1_col], dim=-1),
    ],
    dim=0,
)
train_Y_long = torch.cat([y0, y1], dim=0)

print("train_X_long:", train_X_long.shape)
print("train_Y_long:", train_Y_long.shape)
print(train_X_long[:3])


## 4. Fit `MultiTaskGP`


In [ ]:
mt_model = MultiTaskGP(
    train_X=train_X_long,
    train_Y=train_Y_long,
    task_feature=1,
)

print("raw_train_X:", mt_model.raw_train_X.shape)
print("raw_train_Y:", mt_model.raw_train_Y.shape)
print("supports_mll:", mt_model.supports_mll)

mt_mll = mt_model.make_mll()
fit_gpytorch_mll(mt_mll)


## 5. Posterior for both tasks

When the task feature is omitted from `X`, `output_indices` selects the tasks returned by the posterior.


In [ ]:
test_X = torch.linspace(0.0, 1.0, 200, dtype=dtype).unsqueeze(-1)

mt_model.eval()
with torch.no_grad():
    mt_post = mt_model.posterior(test_X, output_indices=[0, 1])
    mt_mean = mt_post.mean
    mt_var = mt_post.variance

print("posterior mean shape:", mt_mean.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(x0.squeeze(-1), y0.squeeze(-1), label="task 0 observations")
ax.scatter(x1.squeeze(-1), y1.squeeze(-1), label="task 1 observations")
ax.plot(test_X.squeeze(-1), mt_mean[..., 0].squeeze(-1), label="task 0 posterior")
ax.plot(test_X.squeeze(-1), mt_mean[..., 1].squeeze(-1), label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("MultiTaskGP: long-format data")
ax.legend()
plt.show()


## 6. `KroneckerMultiTaskGP`: block-design representation

For the Kronecker model, every task must be observed at the same design points. `train_X` therefore contains only the design variables, while `train_Y` has shape `(n, m)`.


In [ ]:
train_X_block = torch.linspace(0.05, 0.95, 12, dtype=dtype).unsqueeze(-1)
train_Y_block = torch.cat(
    [
        task0(train_X_block),
        task1(train_X_block),
    ],
    dim=-1,
)
train_Y_block = train_Y_block + 0.03 * torch.randn_like(train_Y_block)

print("train_X_block:", train_X_block.shape)
print("train_Y_block:", train_Y_block.shape)


## 7. Fit `KroneckerMultiTaskGP`


In [ ]:
kron_model = KroneckerMultiTaskGP(
    train_X=train_X_block,
    train_Y=train_Y_block,
)

print("raw_train_X:", kron_model.raw_train_X.shape)
print("raw_train_Y:", kron_model.raw_train_Y.shape)
print("raw_train_Yvar:", kron_model.raw_train_Yvar)
print("supports_mll:", kron_model.supports_mll)

kron_mll = kron_model.make_mll()
fit_gpytorch_mll(kron_mll)


## 8. Kronecker posterior


In [ ]:
kron_model.eval()
with torch.no_grad():
    kron_post = kron_model.posterior(test_X)
    kron_mean = kron_post.mean

print("posterior mean shape:", kron_mean.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 0], label="task 0 observations")
ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 1], label="task 1 observations")
ax.plot(test_X.squeeze(-1), kron_mean[..., 0], label="task 0 posterior")
ax.plot(test_X.squeeze(-1), kron_mean[..., 1], label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("KroneckerMultiTaskGP: block design")
ax.legend()
plt.show()


## 9. Choosing between the two

| Model | Training representation | Different X by task? | Typical use |
|---|---|---:|---|
| `MultiTaskGP` | long format, task index in `X` | Yes | tasks measured at different locations |
| `KroneckerMultiTaskGP` | `X[n,d]`, `Y[n,m]` | No | all tasks measured at the same design points |

Both wrappers retain the caller-supplied raw tensors and expose `make_mll()`.
